# Delay Locked Loop Explainer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SiliconJackets/Cac_Spring26/blob/main/DelayLockedLoop.ipynb)

```
Copyright 2026 SiliconJackets @ Georgia Institute of Technology
SPDX-License-Identifier: Apache-2.0
```

|Name|Affiliation| Email |IEEE Member|SSCS Member|
|:--:|:----------:|:----------:|:----------:|:----------:|
|Ethan Huang|Georgia Institute of Technology|ethanhuang@gatech.edu|No|No|
|Member #2|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #3|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #4|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #5|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|
|Member #6|Georgia Institute of Technology|[PLACEHOLDER]@gatech.edu|No|No|

This notebook provides an interactive exploration of a Delay Locked Loop (DLL), illustrating the functionality and design variations of key submodules: the Phase Detector, Controller, DCDL, and Ring Oscillator. It analyzes the behavior and trade-off associated with each component with SPICE simulations. The notebook culminates in a design that measures the delay between two arbitrary clock signals, with the appropriate submodules are chosen through our preceding analysis. The design targets the [open source sky130 PDK](https://github.com/google/skywater-pdk/) by Google and Skywater.

## What is a Delay-Locked Loop
---

A Delay-locked Loop (DLL) is a closed-loop feedback circuit that aligns the phase of an output clock to a reference clock. It shifts the phase of this output clock by delaying it, either through a chain of discrete digital delay cells or through a voltage-controlled analog delay line, until the output edge lines up with the reference edge. Unlike a PLL, a DLL does not generate a new frequency. 

The DLL's core feedback loop consists of three blocks: a Phase Detector, a Controller, and a delay line, for which this notebook uses a Digitally Controlled Delay Line (DCDL). Together, they form a feedback loop that continuously adjusts the delay applied to the output clock until its phase matches the reference, at which point, the system is "locked."

A Ring Oscillator is also implemented to provide a tunable clock source for testing.

> **TODO: Add draw.io diagram (top-level DLL block diagram: CLK_REF -> Phase Detector -> Controller -> DCDL -> CLK_OUT feedback loop)**

Each module section below describes the high-level function, presents each implementation variant with its pros and cons, as evidenced by the spice simulations. 

## Phase Detector
---

The phase detector compares the rising edges of the reference clock (`clk_in`) and the feedback clock (`clk_out`) and produces two single-bit outputs: `up` (reference leads, increase the delay) and `down` (feedback leads, decrease the delay). It is important to note that the phase detector does not measure the magnitude of the phase error, only the direction. 

A bang-bang style of detection is simple, but it means the loop can never settle perfectly still. Near lock, the detector continuously toggles between `up` and `down`, producing jitter. Other phase detector architectures, as explored below, can avoid this intrinsic toggling, but at the cost of added complexity. The choice of phase detector architecture directly impacts the loop's accuracy, stability, and susceptibility to metastability.

> **TODO: Add waveform GIF (phase_detector.gif) showing clk_in, clk_out, up, down signals during lock acquisition and steady state**

> **TODO: Add waveform explanation describing how up/down toggle during acquisition and settle into a limit cycle near lock**

---
### Implementation 1: Single Flip-Flop Detector

> **TODO: Add draw.io diagram (single D flip-flop: clk_out on D, clk_in on CLK, Q driving up/down decode)**

This is perhaps the most trivial phase detector. A single D type flip-flop samples `clk_out` on the rising edge of `clk_in`. If `clk_out` is low at the sampling instant, the feedback clock is trailing behind the reference clock, hence `up` is asserted. Conversely, `down` is asserted if `clk_out` is high at the sampling instant. 

There is no "equal" state; the detector always picks a direction.

| Pros | Cons | 
|----------|----------|
| Extremely low area and complexity. A single flip-flop plus output decode. Smallest power and area footprint of all detectors. | Inherently biased. When clocks are perfectly aligned, the detector still asserts `down` because `clk_out` is sampled as high at the rising edge of `clk_in`. The bias means the loop locks with a small systematic phase offset rather than true zero error. |
| Fully synthesizable, fast timing. Minimal critical path yields high maximum operating frequency (best performance). | No frequency detection capability. |
| | Persistent bang-bang toggling near lock limits achievable jitter performance. |

> **TODO: Add simulation results / waveform figures for single-FF detector**

> **TODO: Add data analysis (bias characterization, jitter measurements)**

---
### Implementation 2: Edge-Order Detector

> **TODO: Add draw.io diagram (two cross-coupled latches driven by clk_in and clk_out, outputs up/down)**

This is an extension of the single flip-flop detector. Each clock edge tries to assert its corresponding output (`up` or `down`), but only succeeds if the other clock's edge has not already arrived. When `clk_in` rises first, `up` is set; when `clk_out` rises first, `down` is set. The opposite edge clears the output, completing the comparison for that cycle. 

| Pros | Cons | 
|----------|----------|
| Minimal logic. Two flip-flop-style blocks with cross-coupled gating. Low area and power, only marginally more than the single-FF detector. | Race conditions when edges arrive nearly simultaneously; both `up` and `down` can briefly assert (observed in aligned-edge test cases). |
| No explicit reset feedback path needed. | When `clk_out` leads `clk_in`, the detector can still incorrectly assert `up` because the first-edge-wins logic lets `clk_in` capture before `down` registers the earlier feedback edge. |
| | No frequency detection capability. Performance is limited by metastability susceptibility when edges are close. |

> **TODO: Add simulation results / waveform figures for edge-order detector**

> **TODO: Add data analysis (measured accuracy across different phase offsets)**

---


### Implementation 3: Phase-Frequency Detector (PFD)

> **TODO: Add draw.io diagram (two FFs with D=1 clocked by clk_in/clk_out, AND gate feeding async reset to both)**

This is the industry-standard architecture. The analysis below will prove why. Two flip-flops are held with D=1. The rising edge of `clk_in` sets the first FF (asserting `up`), and the rising edge of `clk_out` sets the second FF (asserting `down`). An AND gate generates a reset pulse that clears both flip-flops when both outputs are simultaneously high. The width of the `up` or `down` pulse is proportional to the phase difference between the two clocks. This implementation can also detect frequency mismatches; if one clock is faster, its corresponding output will be asserted for longer durations, on average, driving the loop in the correct direction. It is important to note that this implementation is well suited for PLLs, the frequency detection capability is unnecessary in a DLL. Only the phase-detection behavior is used.

| Pros | Cons |
|----------|----------|
| Detects both phase error and frequency error. The only detector here with frequency acquisition capability. | Dead zone. When phase difference is very small, the reset pulse fires almost immediately, producing narrow output pulses that may not be captured reliably by downstream logic. |
| Pulse width encodes error magnitude (useful for analog loops, and provides clean direction for digital bang-bang use). | Slightly more area and power than single-FF or edge-order approaches (two flip-flops, AND gate, and reset feedback path). |
| Well-understood, widely used in PLL/DLL designs. Robust and predictable behavior across PVT corners. | The AND-gate reset path introduces a minimum pulse width constraint that can limit maximum operating frequency. |

> **TODO: Add simulation results / waveform figures for PFD**

> **TODO: Add data analysis (dead zone characterization, frequency acquisition behavior)**

---

### Implementation 4: Sampled XOR Detector

> **TODO: Add draw.io diagram (XOR gate with clk_in/clk_out inputs, output sampled by DFF clocked on clk_in, decode to up/down)**

An XOR gate compares the two clocks and produces a mismatch signal whenever they differ. This mismatch signal is then sampled on the rising edge of the reference clock to determine direction: if the feedback clock is low at the sampling instant, the reference is leading (`up`); if the feedback clock is high, the feedback is leading (`down`); if both clocks agree, both outputs are cleared.

The fundamental problem with this architecture is that the XOR and the direction sample are evaluated at the same instant — the reference clock's rising edge. At that moment, the reference clock is always high, so the XOR reduces to simply checking whether the feedback clock is low or high. When the feedback clock is low, the XOR sees a mismatch and `up` is asserted. When the feedback clock is high, the XOR sees no mismatch and both outputs are cleared — the `down` path is never reached. **The detector can only increase delay or hold; it cannot decrease delay.** This makes the loop fundamentally one-directional.

| Pros | Cons |
|----------|----------|
| Simple combinational front-end (single XOR gate). Minimal area overhead for the detection logic. | One-directional: the detector can only assert `up` or clear both outputs. It cannot assert `down`, so the DLL loop can only increase delay, never decrease it. |
| Naturally produces a "no error" state when clocks are aligned (both outputs cleared). | XOR detects mismatch but not direction; direction is inferred by sampling the feedback clock level, making it duty-cycle and timing dependent. |
| Low power due to minimal switching activity (one XOR gate, one flip-flop). Smallest dynamic power of all detectors. | Not suitable as a standalone detector in practical designs due to the single-sided output limitation. |

> **TODO: Add simulation results / waveform figures for XOR detector**

> **TODO: Add data analysis (duty cycle sensitivity, directional bias measurements)**

## Controller
---

The controller sits between the phase detector and the DCDL. It receives the `up`/`down` signals and maintains an N-bit control word (`ctrl[N-1:0]`) that sets the delay through the DCDL. At its core, every controller variant is a digital accumulator: `ctrl[k+1] = ctrl[k] + up - down`. When `up` is asserted the control word increments (more delay), when `down` is asserted it decrements (less delay), and when both match it holds. The control word is clamped to `[0, 2^N - 1]` to prevent wrap-around. What differentiates the five implementations below is how aggressively they update. The step size, filtering, and mode-switching strategies each trade off lock speed against steady-state jitter. Because the phase detector only provides direction (not magnitude), the controller effectively sets the loop bandwidth: large steps mean fast convergence but more overshoot, small steps mean smooth tracking but slower acquisition.

> **TODO: Add waveform GIF (controller.gif) showing up, down, ctrl[N-1:0] evolving over time during lock acquisition and steady state**

> **TODO: Add waveform explanation describing ctrl ramping during acquisition, settling into limit-cycle toggling near lock**

---

### Implementation 1: Saturating Controller

> **TODO: Add draw.io diagram (adder/subtractor with up/down inputs, saturation clamp, ctrl output register)**

The baseline design. On each clock edge, the control word increments by 1 if `up` is asserted, decrements by 1 if `down` is asserted, and holds otherwise. Hard saturation clamps `ctrl` at 0 and `2^N - 1` to prevent underflow or overflow. On reset, `ctrl` initializes to a parameterized `INIT_CTRL` value (default midpoint of 32 for 6-bit control). The entire update path is a single add/subtract with a compare, one combinational level plus the output register.

| Pros | Cons |
|----------|----------|
| Simplest possible controller - easy to verify and reason about. | Fixed +-1 step means slow convergence when far from lock. |
| Predictable, monotonic response to persistent error. | Near lock, the +-1 toggling produces a limit cycle with 1-LSB jitter that cannot be reduced without external filtering. |
| Smallest area and lowest power of all controllers: one adder/subtractor, one comparator, one register. Industry baseline for comparison. | No adaptability - performance is the same whether the loop is far from or near lock. |

> **TODO: Add simulation results / waveform figures for saturating controller**

> **TODO: Add data analysis (lock time, steady-state jitter amplitude)**

---

### Implementation 2: Filtered Controller

> **TODO: Add draw.io diagram (up/down counters feeding threshold comparators, gating +-1 update to ctrl register)**

Instead of updating `ctrl` on every cycle, this controller requires `FILTER_LEN` consecutive `up` (or `down`) requests before making a single +-1 adjustment. Two internal counters track consecutive `up` and `down` assertions. If the direction changes or both signals are idle, both counters reset to zero. Only when one counter reaches the threshold does `ctrl` update, and then that counter resets. This acts as a simple digital low-pass filter on the phase detector output.

| Pros | Cons |
|----------|----------|
| Significantly reduces jitter near lock; transient noise or single-cycle glitches from the phase detector are ignored. | Slower lock acquisition - the loop must see `FILTER_LEN` consistent cycles before taking any action. |
| Steady-state limit cycle is suppressed because the toggling `up`/`down` pattern resets the counters before threshold is reached. | The `FILTER_LEN` parameter requires tuning: too small and it behaves like the saturating controller, too large and the loop becomes unresponsive. |
| Modest area increase over the saturating controller (two additional counters and comparators). Power consumption is slightly higher due to counter activity, but reduced output toggling saves downstream switching power. | Effective loop bandwidth is reduced by the filter factor, which can be problematic if the clock environment changes rapidly. |

> **TODO: Add simulation results / waveform figures for filtered controller**

> **TODO: Add data analysis (lock time vs FILTER_LEN, jitter reduction compared to saturating)**

---

### Implementation 3: Acquire/Track Controller

> **TODO: Add draw.io diagram (mode FSM selecting between large/small step size, quiet counter triggering mode switch, adder/subtractor with saturation)**

A dual-mode controller. It starts in acquire mode with a large step size (`ACQUIRE_STEP`, default 4) for fast convergence. A quiet counter tracks consecutive idle cycles (no `up` or `down` assertion). Once `QUIET_CYCLES` (default 8) consecutive quiet cycles are observed, indicating the loop is near lock, the controller switches to track mode with a small step size (`TRACK_STEP`, default 1) to minimize jitter. Any new `up` or `down` assertion resets the quiet counter but does not revert the mode. The mode transition is one-way: once in track mode, the controller stays there until reset.

| Pros | Cons |
|----------|----------|
| Fast acquisition (large steps cover the control range quickly) with low steady-state jitter (small steps near lock). | One-way mode switch means the controller cannot re-acquire if the loop is disturbed after locking (e.g., supply noise kicks the delay far from lock). |
| Widely used in industry DLL/PLL designs as a practical balance of speed and stability. | The `QUIET_CYCLES` threshold must be tuned; too aggressive and the controller switches to track mode before actually reaching lock. |
| Moderate area and power - adds only a mode register, quiet counter, and step-size mux over the saturating baseline. | Larger step sizes during acquire mode can cause overshoot, especially if `ACQUIRE_STEP` is too large relative to the DCDL resolution. |

> **TODO: Add simulation results / waveform figures for acquire/track controller**

> **TODO: Add data analysis (acquisition phase duration, jitter comparison between modes)**

---

### Implementation 4: Coarse/Fine Controller

> **TODO: Add draw.io diagram (two separate counters for coarse/fine with mode select, concatenated into ctrl output)**

This controller splits the control word into two fields: an upper coarse field (`COARSE_BITS`, default 3) and a lower fine field (`FINE_BITS`, default 3), concatenated as `ctrl = {coarse, fine}`. It starts in coarse mode, adjusting only the coarse bits for large delay jumps. After `SWITCH_QUIET` (default 8) consecutive quiet cycles, it transitions to fine mode, where only the fine bits are adjusted for precise tuning. Each field saturates independently at its own bounds. This effectively gives the controller two levels of resolution in a single control word.

| Pros | Cons |
|----------|----------|
| High total resolution without needing an extremely wide control word; coarse bits provide range, fine bits provide precision. | More complex than a single-counter design - two independent saturating counters plus mode logic. Larger area than the saturating or filtered controllers. |
| Clean separation of acquisition (coarse) and tracking (fine) phases. | Coarse-to-fine boundary can introduce a jump if the coarse step doesn't land near the correct fine range. |
| Hardware-efficient: the two counters are small and independent. Power overhead is modest since only one counter is active at a time. | Same one-way mode switch limitation as acquire/track - cannot re-acquire after locking. |

> **TODO: Add simulation results / waveform figures for coarse/fine controller**

> **TODO: Add data analysis (resolution improvement, coarse-to-fine transition behavior)**

---

### Implementation 5: Variable-Step Controller

> **TODO: Add draw.io diagram (direction tracker with same_dir_count, step-size lookup table, adder/subtractor with saturation)**

An adaptive controller where the step size grows with repeated error in the same direction. A 4-bit counter (`same_dir_count`) tracks how many consecutive cycles the phase detector has requested the same direction. When the count exceeds `MED_THRESH` (default 4), the step size increases from 1 to `MED_STEP` (default 2). When it exceeds `BIG_THRESH` (default 8), it jumps to `BIG_STEP` (default 4). A direction change resets the counter to 1. This creates nonlinear, accelerating behavior: the loop crawls when error is small (suggesting near-lock) and sprints when error is persistent (suggesting far from lock).

| Pros | Cons |
|----------|----------|
| Fastest convergence of all five controllers - accelerates when far from lock without requiring an explicit mode switch. | More tuning parameters (`MED_THRESH`, `BIG_THRESH`, `MED_STEP`, `BIG_STEP`); poor tuning can cause overshoot or oscillation. |
| Naturally adapts to both acquisition and tracking without separate modes. Can re-accelerate if disturbed, unlike the one-way mode controllers. | Near lock, even a short run of consistent `up` or `down` from the limit cycle can trigger medium steps, potentially increasing jitter compared to a fixed +-1 controller. |
| Moderate area cost - adds a direction register, 4-bit counter, and step-size comparators/mux. Power scales with activity since larger steps mean fewer total update cycles to reach lock. | Most complex controller to verify. Non-linear step behavior makes worst-case jitter harder to bound analytically. |

> **TODO: Add simulation results / waveform figures for variable-step controller**

> **TODO: Add data analysis (convergence speed comparison, overshoot characterization)**

## DCDL (Digitally Controlled Delay Line)
---

The DCDL is the actuator of the DLL. It takes the digital control word from the controller and converts it into a physical propagation delay applied to the clock signal. The input signal `A` enters a chain of delay elements, and the control word `Q` selects how many stages the signal passes through (or which delayed tap is routed to the output `Y`). A larger control value means more delay stages are active, producing a longer delay. The DCDL must provide monotonic, glitch-free delay adjustment across its full control range. Different architectures achieve this using different delay primitives (inverters vs. NAND gates), selection strategies (mux trees vs. bypass logic), and polarity-correction techniques. The five implementations below span the design space from simple inverter chains to dual-chain vernier interpolation.

---

### Implementation 1: Inverter-Based DCDL

> **TODO: Add draw.io diagram (4-stage inverter chain with pairs of inverters per stage, hierarchical 2-level mux tree, output Y)**

> **TODO: Generate waveform GIF for inverter-based DCDL with correct timings**

A 4-stage delay line where each stage consists of two inverters in series (preserving signal polarity). The input `A` propagates through this chain, producing four delayed taps (`tap0` through `tap3`). A 2-bit control word `Q` selects among these taps using a hierarchical mux tree: two first-level muxes each choose between a pair of adjacent taps based on `Q[0]`, and a final mux selects between the two first-level outputs based on `Q[1]`. This gives four selectable delay values with clean, non-inverted output at every tap.

| Pros | Cons |
|----------|----------|
| Straightforward design; the double-inverter stages guarantee consistent polarity at every tap. | Two inverters per stage means more area and power per delay step compared to single-inverter designs. |
| Hierarchical mux tree scales cleanly with more stages. | Mux tree adds its own propagation delay to every path, which may limit minimum achievable delay. |
| Easy to reason about delay linearity. Good baseline for area vs. resolution comparison. | 2-bit control gives only 4 delay steps - coarse resolution. Extending to more steps requires a wider chain and deeper mux tree, increasing area and power quadratically. |

> **TODO: Add simulation results / waveform figures for inverter-based DCDL**

> **TODO: Add data analysis (delay per step, linearity measurement)**

---

### Implementation 2: Conditional Inverter DCDL

> **TODO: Add draw.io diagram (single-inverter chain with 4 stages, mux tree, XNOR gate at output for polarity correction)**

> **TODO: Generate waveform GIF for conditional inverter DCDL with correct timings**

This design uses a single inverter per stage instead of two, halving the chain length for the same number of taps. The trade-off is that odd-indexed taps (`tap0`, `tap2`) produce an inverted signal (odd number of inversions from `A`), while even-indexed taps (`tap1`, `tap3`) are non-inverted. A mux tree identical to Implementation 1 selects the desired tap, and then an XNOR gate at the output conditionally flips the polarity: `Y = mux_out XNOR Q[1]`. When `Q[1]` is 1 the output passes through unchanged; when `Q[1]` is 0 the XNOR inverts it. The intent is to compensate for the varying inversion parity across taps; however, since the inversion parity depends on `Q[0]` (which selects odd vs. even taps within each pair) rather than `Q[1]` (which selects between pairs), the correction is only valid for `Q` values where `Q[0] = Q[1]` (i.e., `Q=00` and `Q=11`). For `Q=01` and `Q=10`, the output polarity is inverted.

| Pros | Cons |
|----------|----------|
| Half the inverters of Implementation 1 for the same number of taps - significantly smaller area and lower power. | The XNOR correction gate adds a fixed delay to every output path, and its own delay variation can affect linearity. |
| Finer delay granularity per stage (each inverter adds one gate delay instead of two). | The polarity correction keys on `Q[1]` but inversion parity depends on `Q[0]`, so polarity is only correct when `Q[0] = Q[1]` (`Q=00`, `Q=11`). For `Q=01` and `Q=10`, the output is inverted. |
| Lower power consumption than the double-inverter chain due to fewer switching elements. | Extending to wider control words requires careful bookkeeping to maintain correct polarity across all tap selections. |

> **TODO: Add simulation results / additional waveform figures for conditional inverter DCDL**

> **TODO: Add data analysis (delay per step vs. Implementation 1, area comparison)**

---

### Implementation 3: Glitch-Free Inverter DCDL

> **TODO: Add draw.io diagram (3-stage inverter chain, one-hot decoder, registered sel_reg, NAND2 gating per tap, NAND tree to output Y)**

> **TODO: Generate waveform GIF for glitch-free DCDL with correct timings**

This variant addresses glitch hazards that can occur when the mux select changes while the signal is propagating. The delay chain consists of 3 inverters producing 4 taps (`tap0` through `tap3`), where `tap0` is the direct input `A` (zero inverter delay) and each subsequent tap adds one inverter delay. Instead of a mux tree, each tap is inverted and then gated with a one-hot select signal using a NAND2 gate. The one-hot select is registered on the clock edge (`sel_reg`), so it only changes synchronously, eliminating combinational glitches during select transitions. The gated tap outputs are combined through a NAND tree (`y0*y1` and `y2*y3`, then the two results combined) to produce the final output. On reset, `sel_reg` defaults to selecting `tap0`.

| Pros | Cons |
|----------|----------|
| Glitch-free by construction; the registered one-hot select ensures no transient paths are created during control word changes. | Requires a clock input (`clk`) and reset (`rst_n`), unlike the purely combinational implementations above. Adds sequential area (register for `sel_reg`). |
| NAND-based gating and combining is area-efficient in standard cell libraries. | The registered select adds one cycle of latency between a control word change and the delay actually updating. This impacts loop response time (performance). |
| Eliminates potential timing violations caused by mid-transition glitches in the other DCDL variants. | The NAND tree path adds fixed delay overhead. More total gates than the simple mux-tree approaches, increasing area and static power. |

> **TODO: Add simulation results / additional waveform figures for glitch-free DCDL**

> **TODO: Add data analysis (glitch characterization vs. Implementations 1 and 2)**

---

### Implementation 4: NAND-Based DCDL

> **TODO: Add draw.io diagram (4-stage chain of nand_dcdl_cells, each cell with in0/in1/ctl inputs showing bypass/propagate logic)**

> **TODO: Generate waveform GIF for NAND-based DCDL with correct timings**

A fundamentally different approach: instead of selecting a tap from a shared chain, each stage is a NAND-based delay cell that can either propagate or bypass the signal. Each cell has three NAND2 gates and one inverter. `nand2_1` gates the propagated input (`in1`) with the inverted control bit (`~ctl`), `nand2_2` gates the bypass input (`in0`) with `ctl`, and `nand2_3` combines the two. When `ctl=1`, the cell passes `in0` (bypass, receiving the input signal `A` directly); when `ctl=0`, it passes `in1` (propagated from the previous stage through the NAND logic, adding delay).

The chain is ordered from cell3 (first in the chain, with `in1` tied to `1'b0`) through cell0 (output `Y = s0`). The first cell with its control bit set to 1 injects signal `A` into the chain; subsequent cells with control bits set to 0 propagate the signal through their NAND gates, adding delay. Cells before the injection point output 0 since cell3's `in1` is tied low. The effective delay scales with the number of propagation stages between the injection point and the output. `dont_touch` and `keep` annotations prevent the synthesizer from optimizing away the intentional delay structure.

| Pros | Cons |
|----------|----------|
| Wider delay range per stage than inverter-based designs, since each NAND cell introduces more gate delay than a single inverter. | Non-linear delay steps; the delay added by enabling one more NAND cell depends on the cell's position in the chain and the state of other cells. |
| Synthesis-robust: `dont_touch` and `keep` annotations preserve the intentional delay structure through the tool flow. | More complex per-cell logic (3 NAND + 1 inverter per stage vs. 1–2 inverters). Higher area and power per stage than inverter-based designs. |
| 4-bit control gives up to 4 stages of propagation delay, offering a wider delay range than the 2-bit inverter DCDLs. | Requires careful control encoding (effectively priority-based: the first `ctl=1` from the MSB injects `A`; arbitrary bit patterns may produce unexpected outputs or stuck-at-zero). |

> **TODO: Add simulation results / additional waveform figures for NAND-based DCDL**

> **TODO: Add data analysis (delay range, per-step nonlinearity)**

---

### Implementation 5: Vernier Crossover DCDL

> **TODO: Add draw.io diagram (dual chain: slow buf_1 chain and fast buf_4 chain with crossover muxes at each stage controlled by Q[i], output from fast_net[STAGES])**

The most sophisticated architecture, targeting fine delay resolution through interpolation between two delay chains with different per-stage delays. A "slow" chain uses `sky130_fd_sc_hd__buf_1` cells (small, slow buffers) and a "fast" chain uses `sky130_fd_sc_hd__buf_4` cells (large, fast buffers). At each of the 16 stages, a 2:1 mux controlled by `Q[i]` can cross the signal from the slow chain into the fast chain. The input `A` enters the slow chain; the fast chain starts at logic-0. When `Q[i]` is asserted at stage `i`, the slow-chain signal crosses over to the fast chain at that point and propagates through the remaining fast-chain stages. The output is taken from the end of the fast chain. The total delay is determined by how many slow stages the signal traverses before crossing over to the fast chain, so more slow stages means more delay.

| Pros | Cons |
|----------|----------|
| Very fine delay resolution; the delay step is the difference between one `buf_1` delay and one `buf_4` delay, which can be much smaller than either buffer's absolute delay. Best performance (finest resolution) of all DCDL variants. | Largest area and power of all five implementations - two full 16-stage buffer chains plus 16 muxes. Significant silicon footprint and highest static/dynamic power. |
| 16-stage design with per-stage control gives high granularity. | Delay linearity depends on matching between the `buf_1` and `buf_4` cell delays across PVT corners. |
| Uses sky130 standard cells directly - ready for physical implementation with no custom cell design needed. | The `fast_net[0] = 1'b0` initialization means the fast chain output is invalid until at least one crossover mux is enabled. Both chains are active simultaneously, increasing power even when only one path carries the signal. |

> **TODO: Add simulation results / waveform figures for vernier DCDL**

> **TODO: Add data analysis (delay resolution measurement, linearity across PVT corners)**

## Ring Oscillator
---

The ring oscillator is not part of the DLL feedback loop itself. It serves as a tunable on-chip clock source for testing and characterization. It generates a free-running clock by feeding the output of an odd-length inverter chain back to its input, creating a self-sustaining oscillation. The oscillation frequency is determined by the total propagation delay around the loop: `f = 1 / (2 * N * t_inv)`, where `N` is the number of inverter stages and `t_inv` is the delay per inverter. This implementation provides frequency selection by tapping the chain at different points.

---

### Implementation 1: Selectable-Length Inverter Ring

> **TODO: Add draw.io diagram (9-inverter chain with feedback from taps n2/n4/n6/n8 through 4:1 mux controlled by sel[1:0] back to clk_out)**

A chain of 9 inverters (inv0 through inv8) is driven from the output `clk_out` back through the chain. A 2-bit select input `sel[1:0]` chooses which tap feeds back to `clk_out` via a combinational case statement:

- `sel = 00` : feedback from `n2` (3 inverters in the loop, highest frequency)
- `sel = 01` : feedback from `n4` (5 inverters, lower frequency)
- `sel = 10` : feedback from `n6` (7 inverters, lower still)
- `sel = 11` : feedback from `n8` (9 inverters, lowest frequency)

Each configuration uses an odd number of inversions, ensuring sustained oscillation. Shorter loops oscillate faster because the total round-trip delay is smaller.

| Pros | Cons |
|----------|----------|
| Simple, compact design - just inverters and a mux. Minimal area and power for an on-chip clock source. | Only one implementation, so no architectural comparison for this module. |
| Provides four discrete frequency options from a single oscillator. | Frequency is highly sensitive to PVT (process, voltage, temperature) variation since it depends entirely on inverter delay. Performance is unpredictable across corners. |
| No external clock input required - useful for standalone testing of the DCDL or full DLL loop. | The `always_comb` feedback style may cause simulation issues in some tools (zero-delay oscillation in RTL sim without gate-level timing). |
| | Frequency selection is coarse - only four options with large gaps between them. No fine-grained frequency tuning capability. |

> **TODO: Add simulation results / waveform figures for ring oscillator at each sel value**

> **TODO: Add data analysis (measured frequency vs. sel setting, PVT sensitivity if available)**

## Practical Examples
---